# Module 4 · Lab 4.1 — End-to-End RAG with LCEL & Citations
## Ground an LLM in a banking policy knowledge base — with traceable sources

---

### Lab overview

Module 3's two labs built the two halves of the RAG front-end: **loaders + chunking**
(Lab 3.1) and **embeddings + semantic search** (Lab 3.2). This lab assembles them into a
complete, auditable **Retrieval-Augmented Generation** pipeline and adds the piece
that makes RAG usable in a regulated domain: **citations**.

We build a banking-policy Q&A assistant end to end:

| Stage | What we do | Builds on |
|---|---|---|
| **Ingest** | Load 6 real policy files from disk | Lab 3.1 (loaders) |
| **Index** | Chunk → embed → store in **FAISS + Chroma** | Lab 3.2 (embeddings) |
| **Retrieve** | Similarity vs **MMR**, with relevance scores | Lab 3.2 (cosine) |
| **Generate** | An **LCEL** chain: `context | prompt | llm` | new |
| **Cite** | Return source docs + inline `[doc N]` markers | new |
| **Guard** | Refuse to answer when the corpus doesn't cover it | new |

### Why citations & the hallucination guard matter
In a bank, a chatbot that invents an interest rate or a credit-score cutoff isn't a
cute bug — it's a compliance incident. So every answer here must (a) come **only**
from retrieved policy text, (b) **cite** the document it used, and (c) say **"I don't
know"** when the policy set doesn't cover the question.

### Learning objectives
1. Assemble a full RAG pipeline with **LangChain LCEL** (`|` pipe + `RunnableParallel`)
2. Index the same corpus in **FAISS** (in-memory) and **Chroma** (persistent) and compare
3. Contrast **similarity** vs **MMR** retrieval and read **relevance scores**
4. Return **grounded answers with source attribution** and inline `[doc N]` citations
5. Add a **hallucination guard** and **persist / reload** your vector stores

### Prerequisites
- `OPENAI_API_KEY` · the workshop VM, Colab, or a local `.env`
- Concepts from Lab 3.2 (embeddings/cosine) and Lab 3.1 (loaders/chunking)

> **Scope note.** This lab covers *basic* retrieval (similarity, MMR). **Advanced**
> strategies — hybrid BM25+vector, parent-child, multi-query, cross-encoder reranking —
> are the subject of this module's **Advanced Retrieval Strategies** take-home lab.

> 🗺️ **Workshop run-list.**
> **Live in session:** §1–13 — the whole pipeline, ending with persist & reload (the corpus Lab 7.4 grades on Day 2)
> **After the session:** §14 FAISS vs Chroma latency check — kept in full so you can finish at your own pace (marked ⏭️).

## 1. Setup — install dependencies

In [1]:
# 📦 Packages this notebook needs. On the workshop VM they are pre-installed from
# the workshop's requirements.txt, so the install line below stays commented out.
# Running somewhere else (Colab, your own laptop)? Uncomment it and run this cell once.
# !pip install -qU chromadb faiss-cpu langchain langchain-chroma langchain-community langchain-core langchain-openai langchain-text-splitters python-dotenv

from importlib.metadata import PackageNotFoundError, version

NOTEBOOK_PACKAGES = [
    "chromadb",
    "faiss-cpu",
    "langchain",
    "langchain-chroma",
    "langchain-community",
    "langchain-core",
    "langchain-openai",
    "langchain-text-splitters",
    "python-dotenv",
]
for package_name in NOTEBOOK_PACKAGES:
    try:
        print(f"{package_name:<26} {version(package_name)}")
    except PackageNotFoundError:
        print(f"{package_name:<26} NOT INSTALLED — uncomment the !pip line above")

chromadb                   1.1.1
faiss-cpu                  1.15.1
langchain                  1.4.3
langchain-chroma           1.0.0
langchain-community        0.4.2
langchain-core             1.6.6
langchain-openai           1.6.6
langchain-text-splitters   1.1.2
python-dotenv              1.2.3


> ⚠️ **One expected warning.** The first cell that imports from `langchain_community` prints a
> `DeprecationWarning` saying the package *"is being sunset"*. LangChain stopped maintaining its
> community-integrations package in May 2026. The classes this lab uses from it — `DirectoryLoader`, `FAISS`, `TextLoader` —
> have no maintained LangChain replacement yet, so the workshop keeps them, pinned at the version
> tested here (0.4.2): the code works as shown. Integrations that did move to their own packages
> are imported from those — Chroma from `langchain-chroma`, Tavily from `langchain-tavily`.

## 2. Load your API key

In [2]:
import os

# Where to create each key. The trainer supplies OPENAI_API_KEY on the workshop VM;
# you create any other key yourself and add it to the workshop's .env file as KEY_NAME=value.
KEY_SIGNUP_PAGES = {
    "OPENAI_API_KEY": "supplied on the workshop VM (elsewhere: https://platform.openai.com/api-keys)",
    "TAVILY_API_KEY": "https://app.tavily.com (free tier)",
    "GROQ_API_KEY": "https://console.groq.com/keys (free tier)",
    "WEATHER_API_KEY": "https://www.weatherapi.com/signup.aspx (free tier)",
    "LANGSMITH_API_KEY": "https://smith.langchain.com -> Settings -> API Keys (free tier)",
    "ANTHROPIC_API_KEY": "https://console.anthropic.com/settings/keys",
    "GEMINI_API_KEY": "https://aistudio.google.com/app/apikey",
}


def load_keys(required):
    """Load API keys from Colab Secrets (on Colab) or a local .env file (elsewhere).

    Prints one line per key, then stops with instructions if a required key is missing.

    Args:
        required: the environment-variable names this lab needs.
    """
    try:
        from google.colab import userdata          # running on Google Colab
        for key_name in required:
            try:
                os.environ[key_name] = userdata.get(key_name)
            except Exception:
                pass                                 # not set in Colab Secrets
        source = "Colab Secrets"
    except ImportError:                              # the workshop VM, or any local machine
        from dotenv import find_dotenv, load_dotenv
        load_dotenv(find_dotenv(usecwd=True))        # searches upward for the workshop's .env file
        source = ".env / environment"
    print(f"\U0001F511 Key source: {source}")
    for key_name in required:
        print(f"   {key_name:<22} {'✅' if os.environ.get(key_name) else '❌ MISSING'}")
    missing_keys = [key_name for key_name in required if not os.environ.get(key_name)]
    if missing_keys:
        instructions = "\n".join(
            f"  {key_name}: {KEY_SIGNUP_PAGES.get(key_name, 'see README.md in the workshop folder')}"
            for key_name in missing_keys)
        raise RuntimeError(
            "This lab needs API keys that are not set yet. Create each one, add it to\n"
            "the workshop's .env file as KEY_NAME=value, then re-run this cell:\n" + instructions)

load_keys(["OPENAI_API_KEY"])

🔑 Key source: .env / environment
   OPENAI_API_KEY         ✅


## 3. Imports

Note the vector-store imports: **FAISS** from `langchain_community`, but **Chroma**
from the dedicated **`langchain_chroma`** package — the current, maintained home for
the Chroma integration (the older `langchain_community.vectorstores.Chroma` is
deprecated).

In [3]:
import time, shutil, statistics
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma                      # current Chroma integration
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown

EMBED_MODEL = "text-embedding-3-small"   # 1536-dim (see Lab 3.2)
CHAT_MODEL  = "gpt-4.1-mini"             # fast, non-reasoning, supports temperature
print("✅ Imports ready")

C:\Users\participant\AppData\Local\Temp\ipykernel_27196\2752199665.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


✅ Imports ready


## 4. Load the shared corpus (pre-provided)

A real RAG system ingests **files**, not Python strings. The banking knowledge base is
**pre-provided** in `./docs` — the *same* files Lab 4.2 and Lab 7.4 use; we do **not** regenerate it.
We load it with a LangChain loader, exactly the front-end you built in Lab 3.1.

In [4]:
import os
# The banking knowledge base is PRE-PROVIDED in ./docs (shared with Lab 4.2 and Lab 7.4) — not regenerated here.
# On Colab, upload the docs/ folder next to this notebook.
assert os.path.isdir("docs"), "❌ Missing ./docs — the shared banking-policy corpus."
print("✅ Corpus present:", sorted(file_name for file_name in os.listdir("docs") if file_name.endswith(".md")))

✅ Corpus present: ['commercial_lending.md', 'credit_cards.md', 'deposit_accounts.md', 'fraud_security.md', 'investments_retirement.md', 'mortgage_lending.md', 'retail_loans.md', 'service_codes_reference.md', 'tax_forms_reference.md']


## 5. Load the files (callback to Lab 3.1)

`DirectoryLoader` walks the folder and applies `TextLoader` to each `.md` file —
one `Document` per file. `TextLoader` only records the file path in metadata, so we
enrich each doc with a clean `title` and a `category` (that metadata becomes our
**citations** downstream).

In [5]:
loader = DirectoryLoader(
    "docs", glob="*.md",
    loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"},
)
raw_docs = loader.load()

for document in raw_docs:
    file_name = os.path.basename(document.metadata["source"])     # e.g. credit_card_policy.md
    document.metadata["title"] = file_name.removesuffix(".md").replace("_", " ").title()
    document.metadata["category"] = "banking_policy"

print(f"✅ Loaded {len(raw_docs)} documents")
for document in raw_docs:
    print(f"   {document.metadata['title']:<38} {len(document.page_content):>5} chars")

✅ Loaded 9 documents
   Commercial Lending                       726 chars
   Credit Cards                            1000 chars
   Deposit Accounts                         945 chars
   Fraud Security                           884 chars
   Investments Retirement                   906 chars
   Mortgage Lending                        1015 chars
   Retail Loans                            1012 chars
   Service Codes Reference                 1071 chars
   Tax Forms Reference                     1131 chars


## 6. Chunk the documents

Six whole policies are too coarse to retrieve against — a query about "FHA down
payment" should pull the *paragraph*, not the entire mortgage policy. We split with
`RecursiveCharacterTextSplitter` (the Lab 3.1 default): 1000-char chunks, 200-char
overlap. Metadata (including our `title`) propagates to every chunk.

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200,
    separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " ", ""],
)
chunks = splitter.split_documents(raw_docs)

print(f"✅ {len(raw_docs)} documents → {len(chunks)} chunks")
print(f"\nExample chunk (from '{chunks[0].metadata['title']}'):")
print(chunks[0].page_content[:200], "...")

✅ 9 documents → 13 chunks

Example chunk (from 'Commercial Lending'):
# Commercial Lending Policy  (Policy Ref: POL-COM-2024-01)

## Term Loans
Eligibility: personal credit score 680+, business credit score (Paydex) 75+, 2+ years in
business, and 500,000 minimum annual  ...


## 7. Embed & index — FAISS *and* Chroma

We embed every chunk (Lab 3.2's `text-embedding-3-small`) and load the vectors into
**two** stores so you can feel the trade-off:

| Store | Lives | Best when |
|---|---|---|
| **FAISS** | In-memory (save/load to disk manually) | Fastest search; you manage persistence |
| **Chroma** | On-disk (auto-persisted) | You want a queryable DB that survives restarts |

*(We wipe any old Chroma directory first so re-running the cell doesn't insert the
same chunks twice — a classic persistent-store gotcha.)*

In [7]:
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

# FAISS — in-memory
start_time = time.time()
faiss_store = FAISS.from_documents(chunks, embeddings)
print(f"✅ FAISS  indexed {faiss_store.index.ntotal} vectors in {time.time()-start_time:.2f}s")

# Chroma — persistent (wipe first to avoid duplicate inserts on re-run)
shutil.rmtree("./chroma_banking", ignore_errors=True)
start_time = time.time()
chroma_store = Chroma.from_documents(
    chunks, embeddings,
    persist_directory="./chroma_banking",
    collection_name="banking_policies",
)
print(f"✅ Chroma indexed {chroma_store._collection.count()} vectors in {time.time()-start_time:.2f}s (persisted to ./chroma_banking)")

✅ FAISS  indexed 13 vectors in 3.57s


✅ Chroma indexed 13 vectors in 2.06s (persisted to ./chroma_banking)


## 8. Retrieval strategies — similarity vs MMR

A **retriever** turns a query into an embedding and returns the closest chunks. Two
strategies to know now (advanced ones are in this module's take-home lab):

| Strategy | Returns | Use when |
|---|---|---|
| **Similarity** | Top-k nearest chunks | Default; fast and usually enough |
| **MMR** (Maximal Marginal Relevance) | Top-k that are relevant **and diverse** | The top hits are near-duplicates and you want broader coverage |

MMR pulls `fetch_k` candidates, then greedily picks `k` that balance relevance
against redundancy.

In [8]:
faiss_similarity  = faiss_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
chroma_similarity = chroma_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
chroma_mmr        = chroma_store.as_retriever(search_type="mmr", search_kwargs={"k": 4, "fetch_k": 20})

def compare_retrievers(query, retrievers):
    """Print, for each named retriever, the titles of the documents it returns for `query`."""
    print(f'🔍 "{query}"\n')
    for name, retriever in retrievers.items():
        titles = [document.metadata.get("title", "?") for document in retriever.invoke(query)]
        print(f"  {name:<26} → {titles}")

compare_retrievers(
    "What credit score and down payment do I need for a mortgage?",
    {"FAISS (similarity)": faiss_similarity,
     "Chroma (similarity)": chroma_similarity,
     "Chroma (MMR)": chroma_mmr},
)

🔍 "What credit score and down payment do I need for a mortgage?"



  FAISS (similarity)         → ['Mortgage Lending', 'Commercial Lending', 'Mortgage Lending', 'Retail Loans']


  Chroma (similarity)        → ['Mortgage Lending', 'Commercial Lending', 'Mortgage Lending', 'Retail Loans']


  Chroma (MMR)               → ['Mortgage Lending', 'Mortgage Lending', 'Tax Forms Reference', 'Service Codes Reference']


### Relevance scores — how confident is each hit?
`similarity_search_with_relevance_scores` returns a normalized score in `[0, 1]`
(higher = closer). In production you'd set a floor (e.g. drop chunks < 0.7) to keep
weak matches out of the prompt.

In [9]:
query = "What are the fees for a business checking account?"
print(f'🔍 "{query}"\n')
print(f"{'Score':<9}{'Source':<34}{'Preview'}")
print("-" * 90)
for document, score in chroma_store.similarity_search_with_relevance_scores(query, k=4):
    print(f"{score:<9.3f}{document.metadata.get('title','?'):<34}{document.page_content[:40].strip()}…")

🔍 "What are the fees for a business checking account?"

Score    Source                            Preview
------------------------------------------------------------------------------------------


0.072    Fraud Security                    # Fraud Prevention & Account Security  (…
0.069    Deposit Accounts                  # Deposit Account Terms  (Policy Ref: PO…
0.056    Commercial Lending                # Commercial Lending Policy  (Policy Ref…
0.048    Credit Cards                      # Credit Card Products  (Policy Ref: POL…


## 9. The RAG chain — LCEL with source attribution

Now the generation half. We compose an **LCEL** chain that:
1. runs the retriever on the question,
2. formats the retrieved chunks into a numbered context block (`[doc 1] …`),
3. fills a **context-only** prompt, and
4. calls the LLM.

We wrap it in `RunnableParallel` so the chain returns the **answer *and* the source
documents** side by side — the auditable shape you want in a regulated setting.

In [10]:
chat_model = ChatOpenAI(model=CHAT_MODEL, temperature=0)   # temperature 0 → factual, repeatable

REFUSAL = "I don't have that information in the provided policy documents."

qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a precise banking-policy assistant. Answer using ONLY the context below.\n"
     "Guidelines:\n"
     "1. Include exact figures (rates, limits, fees, scores) when the context has them.\n"
     "2. After each factual claim, cite the supporting source inline as [doc N].\n"
     f'3. If the context does not contain the answer, reply EXACTLY: "{REFUSAL}"\n'
     "4. Never use outside knowledge."),
    ("human", "Question: {question}\n\nContext:\n{context}\n\nAnswer:"),
])

def format_docs(documents):
    """Render retrieved chunks as a numbered context block the LLM can cite by [doc N]."""
    return "\n\n".join(
        f"[doc {index+1}] (Source: {document.metadata.get('title', 'Unknown')})\n{document.page_content}"
        for index, document in enumerate(documents)
    )

def build_rag_chain(retriever):
    """LCEL chain returning BOTH the answer and the source documents."""
    return RunnableParallel(
        {"docs": retriever, "question": RunnablePassthrough()}
    ).assign(
        answer=(
            {"context": lambda inputs: format_docs(inputs["docs"]),
             "question": lambda inputs: inputs["question"]}
            | qa_prompt | chat_model | StrOutputParser()
        )
    )

rag_chain = build_rag_chain(faiss_similarity)
print("✅ RAG chain ready (retrieve → format → prompt → llm, returning answer + sources)")

✅ RAG chain ready (retrieve → format → prompt → llm, returning answer + sources)


In [11]:
def ask_question(question, chain=None):
    """Run the RAG chain and pretty-print the grounded answer + its source map."""
    chain = chain or rag_chain
    result = chain.invoke(question)
    print("=" * 84)
    print(f"💬 {question}\n")
    display(Markdown(result["answer"]))
    print("\n📚 Sources the model could cite:")
    for index, document in enumerate(result["docs"], 1):
        print(f"   [doc {index}] {document.metadata.get('title','?')}  ·  {document.page_content[:70].strip()}…")
    print("=" * 84)

ask_question("What credit score is required for a commercial loan, and what documents are needed?")

💬 What credit score is required for a commercial loan, and what documents are needed?



The credit score required for a commercial loan (term loan) is a personal credit score of 680 or higher and a business credit score (Paydex) of 75 or higher. Additionally, the business must have been operating for 2 or more years and have a minimum annual revenue of 500,000. Loan amounts range from 50,000 to 5,000,000 with terms of 5 to 25 years [doc 1].

The documents needed are not explicitly listed in the provided policy documents. However, eligibility criteria imply that documentation proving personal and business credit scores, business operation duration, and annual revenue would be required. Also, for loans above 250,000, a personal guarantee is required [doc 1].

I don't have that information in the provided policy documents regarding the exact list of documents needed.


📚 Sources the model could cite:
   [doc 1] Commercial Lending  ·  # Commercial Lending Policy  (Policy Ref: POL-COM-2024-01)

## Term Lo…
   [doc 2] Retail Loans  ·  # Retail Loan Products  (Policy Ref: POL-RTL-2024-07)

## Personal Loa…
   [doc 3] Mortgage Lending  ·  # Mortgage Lending Policy  (Policy Ref: POL-MTG-2024-03)

## Conventio…
   [doc 4] Retail Loans  ·  ## Education Loan
Covers tuition, living expenses, and equipment. Repa…


## 10. Inline citations you can audit

Because `format_docs` numbers the context as `[doc 1] … [doc N]` and the prompt
tells the model to cite after every claim, the answer text now carries **inline
`[doc N]` markers** that map straight back to the source list. That traceability —
"which document backs this sentence?" — is what turns a demo into something a
compliance team will sign off on.

In [12]:
for question in [
    "How much can I contribute to a Traditional IRA in 2024, and what CD rate can I get?",
    "What is the annual fee and rewards rate on the Premium Rewards card?",
]:
    ask_question(question)
    time.sleep(1)

💬 How much can I contribute to a Traditional IRA in 2024, and what CD rate can I get?



You can contribute up to $7,000 to a Traditional IRA in 2024, or $8,000 if you are age 50 or older [doc 1].

For Certificates of Deposit (CDs), the rates are as follows:
- 1-year CD: 5.00% APY
- 3-year CD: 4.85% APY
- 5-year CD: 4.60% APY

Early withdrawal from CDs forfeits 3 to 12 months of interest depending on the tenure [doc 1][doc 4].


📚 Sources the model could cite:
   [doc 1] Investments Retirement  ·  # Investment & Retirement Services  (Policy Ref: POL-INV-2024-09)

##…
   [doc 2] Mortgage Lending  ·  # Mortgage Lending Policy  (Policy Ref: POL-MTG-2024-03)

## Conventio…
   [doc 3] Tax Forms Reference  ·  ## TDS on Interest
Banks deduct 10% TDS on interest above 40,000 per y…
   [doc 4] Deposit Accounts  ·  # Deposit Account Terms  (Policy Ref: POL-DEP-2024-02)

## Savings Acc…


💬 What is the annual fee and rewards rate on the Premium Rewards card?



The Premium Rewards Card has an annual fee of 95. The rewards rate is 3% on travel, 2% on dining, and 1% on all other purchases. Additionally, there is a sign-up bonus of 500 after spending 3,000 in 90 days [doc 1].


📚 Sources the model could cite:
   [doc 1] Credit Cards  ·  # Credit Card Products  (Policy Ref: POL-CC-2024-05)

## Premium Rewar…
   [doc 2] Retail Loans  ·  # Retail Loan Products  (Policy Ref: POL-RTL-2024-07)

## Personal Loa…
   [doc 3] Deposit Accounts  ·  # Deposit Account Terms  (Policy Ref: POL-DEP-2024-02)

## Savings Acc…
   [doc 4] Fraud Security  ·  # Fraud Prevention & Account Security  (Policy Ref: POL-SEC-2024-06)…


## 11. The hallucination guard 🛡️

The real test of a grounded system is what it does with a question the corpus
**can't** answer. A naive LLM will happily invent a plausible-sounding rate — in a
bank, that's a regulatory incident. Our context-only prompt + fixed refusal string
means the system should decline instead. We probe it with off-corpus queries:

In [13]:
off_corpus = [
    "What is the bank's cryptocurrency trading policy?",   # no crypto doc
    "What are the premiums for pet insurance?",            # no insurance-product doc
    "Can I open a Swiss numbered account here?",           # not covered
]
for question in off_corpus:
    result = rag_chain.invoke(question)
    is_refused = REFUSAL.lower() in result["answer"].lower()
    print(f'{"✅ refused" if is_refused else "⚠️ ANSWERED (leak!)"} — "{question}"')
    print(f'      → {result["answer"][:110]}\n')

✅ refused — "What is the bank's cryptocurrency trading policy?"
      → I don't have that information in the provided policy documents.



✅ refused — "What are the premiums for pet insurance?"
      → I don't have that information in the provided policy documents.



✅ refused — "Can I open a Swiss numbered account here?"
      → I don't have that information in the provided policy documents.



If every line reads **✅ refused**, the guard holds: the assistant stays inside
its knowledge base. This is the single most important behaviour to verify before
putting RAG in front of customers.

## 12. Does retrieval strategy change the answer? (similarity vs MMR)

Same question, two retrievers feeding the same chain. When the top similarity hits
are redundant, MMR's diversity can surface a fact the similarity chain missed.

In [14]:
rag_chain_mmr = build_rag_chain(chroma_mmr)
question = "What account options pay interest, and what balances do they require?"

print("──────── SIMILARITY retriever ────────")
display(Markdown(rag_chain.invoke(question)["answer"]))
print("\n──────── MMR retriever ────────")
display(Markdown(rag_chain_mmr.invoke(question)["answer"]))

──────── SIMILARITY retriever ────────


The account options that pay interest and their required balances are:

1. SB-GOLD-001 — premium savings account: requires a minimum balance of 3,000,000 and pays 4.0% APY.
2. SB-REG-002 — regular savings account: requires a minimum balance of 10,000 and pays 2.75% APY.
3. Fixed Deposits: various tenures from 7 days to 10 years with rates such as 5.00% APY for 1-year, 4.85% for 3-year, and 4.60% for 5-year deposits.
4. Certificates of Deposit: similar rates as fixed deposits, 5.00% APY for 1-year, 4.85% for 3-year, and 4.60% for 5-year terms.

The zero-balance salary account (SB-ZERO-003) does not pay interest. Current accounts do not specify interest payments [doc 1][doc 2].


──────── MMR retriever ────────


The account options that pay interest and their required balances are:

- SB-GOLD-001 premium savings account: requires a minimum balance of 3,000,000 and pays 4.0% APY.
- SB-REG-002 regular savings account: requires a minimum balance of 10,000 and pays 2.75% APY.
- Fixed Deposits: various tenures from 7 days to 10 years with rates such as 5.00% APY for 1-year, 4.85% for 3-year, and 4.60% for 5-year deposits. The FD-TAX-SAVER product has a 5-year lock-in [doc 1].

## 13. Persist & reload

Embeddings cost money and time — you don't want to rebuild the index every run.
**FAISS** saves/loads a folder; **Chroma** is already on disk and just needs to be
reopened by pointing at the same directory + collection.

In [15]:
# FAISS: save and reload
faiss_store.save_local("banking_faiss_index")
loaded_faiss = FAISS.load_local(
    "banking_faiss_index", embeddings,
    allow_dangerous_deserialization=True,      # you trust this file (you wrote it)
)
print(f"✅ FAISS reloaded — {loaded_faiss.index.ntotal} vectors")
print("   sanity check:", loaded_faiss.similarity_search("VA loan funding fee", k=1)[0].metadata["title"])

# Chroma: reopen the persisted directory (no re-embedding)
reopened_chroma = Chroma(
    persist_directory="./chroma_banking",
    embedding_function=embeddings,
    collection_name="banking_policies",
)
print(f"✅ Chroma reopened — {reopened_chroma._collection.count()} vectors")

✅ FAISS reloaded — 13 vectors


   sanity check: Mortgage Lending
✅ Chroma reopened — 13 vectors


> ⏭️ **After the session.** This section is not run live in the workshop. Work through it afterwards, top to bottom.

## 14. FAISS vs Chroma — a quick latency check

Both are fast at this scale; the point is the **method**, not the millisecond. FAISS
(pure in-memory) usually edges out Chroma (which crosses a DB boundary) — the gap
matters only at scale, and Chroma buys you persistence in exchange.

In [16]:
bench_queries = ["credit score requirements", "mortgage rates", "IRA contribution limit",
                 "business checking fees", "how to report fraud"]

def mean_latency_ms(store):
    """Return the mean similarity-search latency of `store` over the benchmark queries, in ms."""
    latencies_ms = []
    for query in bench_queries:
        start_time = time.time(); store.similarity_search(query, k=4); latencies_ms.append((time.time()-start_time)*1000)
    return statistics.mean(latencies_ms)

faiss_mean_ms, chroma_mean_ms = mean_latency_ms(faiss_store), mean_latency_ms(chroma_store)
print(f"FAISS  mean: {faiss_mean_ms:6.2f} ms/query")
print(f"Chroma mean: {chroma_mean_ms:6.2f} ms/query")
faster = "FAISS" if faiss_mean_ms < chroma_mean_ms else "Chroma"
print(f"→ {faster} is {max(faiss_mean_ms,chroma_mean_ms)/min(faiss_mean_ms,chroma_mean_ms):.1f}× faster here (both fine at this corpus size)")

FAISS  mean: 387.66 ms/query
Chroma mean: 438.07 ms/query
→ FAISS is 1.1× faster here (both fine at this corpus size)


## 15. Conclusion & what's next

| You built | Why it matters |
|---|---|
| Files → loader → chunks → vectors | The full ingestion front-end (Lab 3.1 + Lab 3.2, assembled) |
| FAISS **and** Chroma, side by side | Speed vs persistence — a real deployment choice |
| Similarity vs MMR + relevance scores | Tune *what* context reaches the LLM |
| LCEL chain returning answer **+ sources** | Grounded generation you can audit |
| Inline `[doc N]` citations | Trace every claim back to a policy document |
| Hallucination guard | Refuse rather than invent — the compliance-critical behaviour |
| Persist / reload | Don't pay to re-embed every run |

**The big picture.** You now have a complete, grounded, citable RAG system — the
backbone of the **Agentic RAG** you'll give an agent as a *tool* in Module 6's
Financial Analyst lab — and the corpus Lab 7.4's self-correcting graph grades.

### What's still missing (→ the Advanced Retrieval take-home lab)
Our retriever is single-strategy. When plain similarity isn't enough, you reach for
**advanced retrieval**: hybrid **BM25 + vector**, **HyDE**,
**cross-encoder re-ranking**, and **contextual chunks**. That's this module's take-home lab.

### Try on your own
1. Drop `k` to 1 — where do answers get worse, and which questions still work?
2. Add a relevance-score floor (skip chunks < 0.7) inside `format_docs` and re-test the guard.
3. Point the loader at a folder of **real PDFs** (Lab 3.1's `PyMuPDFLoader`) and re-index.
4. Swap the FAISS retriever for `search_type="mmr"` and diff the citations.

---
**Next → Lab 4.2 — Conversational RAG** (memory, compression and persistence on top of this chain). Advanced retrieval — hybrid BM25 + vector, HyDE, re-ranking — is this module's take-home lab.